# Szemerédi-Trotter Constructions

In [ ]:
#@title Verification code

import numpy as np
import math
from numba import njit
from typing import Tuple, List


@njit
def extended_gcd(a: int, b: int) -> Tuple[int, int, int]:
  """Extended Euclidean Algorithm: returns g, x, y such that ax + by = g."""
  if a == 0:
    return b, 0, 1
  g, x1, y1 = extended_gcd(b % a, a)
  x = y1 - (b // a) * x1
  y = x1
  return g, x, y


@njit
def count_solutions_in_box(a: int, b: int, c: int, W: int, H: int) -> int:
  """Counts integer solutions to ax + by = c with 0 <= x < W and 0 <= y < H."""
  if W <= 0 or H <= 0:
    return 0
  if a == 0 and b == 0:
    return W * H if c == 0 else 0

  if a == 0:
    if b != 0 and c % b == 0:
      py = c // b
      return W if 0 <= py < H else 0
    return 0
  if b == 0:
    if a != 0 and c % a == 0:
      px = c // a
      return H if 0 <= px < W else 0
    return 0

  g, x0_base, y0_base = extended_gcd(a, b)
  if c % g != 0:
    return 0

  x0 = x0_base * (c // g)
  y0 = y0_base * (c // g)
  b_g = b // g
  a_g = a // g

  k_min = -np.inf
  k_max = np.inf

  if b_g > 0:
    k_min = max(k_min, np.ceil(-x0 / b_g))
    k_max = min(k_max, np.floor((W - 1 - x0) / b_g))
  else:
    k_min = max(k_min, np.ceil((W - 1 - x0) / b_g))
    k_max = min(k_max, np.floor(-x0 / b_g))

  if a_g > 0:
    k_min = max(k_min, np.ceil((y0 - (H - 1)) / a_g))
    k_max = min(k_max, np.floor(y0 / a_g))
  else:
    k_min = max(k_min, np.ceil(y0 / a_g))
    k_max = min(k_max, np.floor((y0 - (H - 1)) / a_g))

  if k_max < k_min:
    return 0
  return int(k_max - k_min) + 1


@njit
def calculate_score_numba(lines: np.ndarray, H: int, n: int) -> float:
  """Calculates raw incidences using O(log n) formula per line."""
  if H <= 0:
    return 0.0
  W = n // H
  incidence_count = 0

  for i in range(lines.shape[0]):
    a, b, c = lines[i, 0], lines[i, 1], lines[i, 2]
    incidence_count += count_solutions_in_box(a, b, c, W, H)

    remaining_points = n - H * W
    if remaining_points > 0:
      c_new = c - b * H
      incidence_count += count_solutions_in_box(a, 0, c_new, remaining_points, 1)

  return float(incidence_count)


def calculate_score(construction, n):
  """Calculates the number of incidences for a given construction."""
  if not (isinstance(construction, tuple) and len(construction) == 2):
    return -np.inf

  H, lines = construction
  if H <= 0 or len(lines) != n:
    return -np.inf

  unique_lines_set = set()
  for a, b, c in lines:
    if a == 0 and b == 0:
      continue
    common_divisor = math.gcd(math.gcd(a, b), c)
    if common_divisor != 0:
      a //= common_divisor
      b //= common_divisor
      c //= common_divisor
    if a < 0 or (a == 0 and b < 0):
      a, b, c = -a, -b, -c
    unique_lines_set.add((a, b, c))

  unique_lines_list = list(unique_lines_set)
  if not unique_lines_list:
    lines_arr = np.array([], dtype=np.int64).reshape(0, 3)
  else:
    lines_arr = np.array(unique_lines_list, dtype=np.int64)

  return calculate_score_numba(lines_arr, H, n)

In [ ]:
#@title Initial program

import numpy as np
from typing import Tuple, List


def search_for_best_construction(
    n: int,
) -> Tuple[int, List[Tuple[int, int, int]]]:
  """Implements the Elekes construction for generating points and lines.

  The point set is a tall grid (H ~ n^(2/3), W ~ n^(1/3)).
  The line set L consists of lines y = ax + b, where a and b are small integers.
  """
  if n <= 1:
    return 1, [(1, -1, 0)]

  H = int(np.round(n ** (2 / 3)))
  if H == 0:
    H = 1

  num_slopes = int(np.round(n ** (1 / 3)))
  if num_slopes == 0:
    num_slopes = 1

  num_intercepts_per_slope = n // num_slopes if num_slopes > 0 else n

  lines = []
  for a in range(1, num_slopes + 1):
    for b in range(1, num_intercepts_per_slope + 1):
      if len(lines) < n:
        lines.append((a, -1, -b))

  i = 1
  while len(lines) < n:
    lines.append((1, -1, -i))
    i += 1

  return H, lines


# Verify the solution
ns_to_test = [5, 9, 16, 20, 100, 1000]
for n in ns_to_test:
  construction = search_for_best_construction(n)
  raw_score = calculate_score(construction, n)
  sota_scaling = 1.27 * (n ** (4 / 3)) if n > 100 else n
  print(f"n={n}: incidences={raw_score:.0f}, ratio={raw_score/sota_scaling:.4f}")

**Prompt used**

Act as an expert in combinatorial geometry and algorithm design. Your task is to solve the following problem.

Problem Statement:
The Szemerédi-Trotter theorem provides an upper bound on the number of incidences between points and lines in the Euclidean plane. Your task is to find a configuration of n points and n lines that achieves a high number of incidences, approaching the theoretical maximum.

Your Task:
You must write a Python function calle search_for_best_construction that, for a parameter n, returns a construction with n points and n lines that has as many point-line incidences as possible. The point set will always be a rectangular subset of the integer lattice. "Best" means it maximizes the number of point-line incidences. Your function should return the construction as a tuple containing two elements: (H, lines).

H: An integer representing the height of the point grid. The set of n points is implicitly defined by n and H as a grid of width W = n // H with a possible remainder row.

lines: A list of n tuples (a, b, c) of three integers representing lines of the form ax + by = c.

The input to your function will be an integer n.

Evaluation:
Your proposed construction will be evaluated by a function which returns the raw number of point-line incidences. The overall evaluation normalizes this score using the state-of-the-art bound of approximately 1.27 * n^(4/3). A higher normalized score means your construction is better.

Your function search_for_best_construction must deterministically generate a construction for any given n.

Line Representation:
To avoid floating-point errors, you must use the integer equation ax + by = c to represent lines. Your final list of lines should be in this (a, b, c) format.

Big important hint: you will be evaluated against a wide range of small and very large prime values of n, so you MUST try to find a general solution to the problem. I would strongly encourage you to try to find a general solution. Your program will be evaluated on some very large values of n -- try to find the pattern that works for all n.

Second hint: the previous solution provided in this prompt is good, but still not optimal, much better configurations are possible. The patterns you have to discover are not hard, you can definitely improve it, it is not beyond your capabilities. DO NOT go for the same solution as the previous one. Always try to find a better pattern, don't be scared of the difficult sounding problem, once you see the solution you'll realise it wasn't hard at all. Good luck, I believe in you, but you also have to believe in yourself!

Good luck!

**Another prompt used**

Act as an expert in combinatorial geometry and algorithm design. Your task is to solve the following problem.

Problem Statement
The Szemerédi-Trotter theorem provides an upper bound on the number of incidences between points and lines in the Euclidean plane. Your task is to find a configuration of n points and n lines that achieves a high number of incidences, approaching the theoretical maximum.

Your Task
You must write a Python function called search_for_best_construction that, for a parameter n, returns a construction with n points and n lines that has as many point-line incidences as possible. "Best" means it maximizes the number of point-line incidences. Your function should return the construction as a tuple containing two elements: (points, lines).

points: A list of n tuples (x, y) representing the coordinates of the chosen points. All points must be on the integer lattice and lie within the n x n square, i.e., 0 <= x < n and 0 <= y < n.

lines: A list of n tuples (a, b, c) of three integers representing lines of the form ax + by = c.

The input to your function will be an integer n.

Evaluation
Your proposed construction will be evaluated by a function which returns the raw number of point-line incidences. The overall evaluation normalizes this score using the state-of-the-art bound of approximately 1.27 * n^(4/3). A higher normalized score means your construction is better.

Your function search_for_best_construction must deterministically generate a construction for any given n.

Line Representation
To avoid floating-point errors, you must use the integer equation ax + by = c to represent lines. Your final list of lines should be in this (a, b, c) format.

Hints
Big important hint: You will be evaluated against a wide range of small and very large values of n, so you MUST try to find a general solution to the problem. Your program will be evaluated on some very large values of n—try to find the pattern that works for all n.

Second hint: The initial provided construction (based on the Elekes point set) is good, but still not optimal; much better configurations are possible. The patterns you have to discover are not hard. You can definitely improve it; it is not beyond your capabilities. DO NOT go for the same solution as the previous one. Always try to find a better pattern. Don't be scared of the difficult-sounding problem; once you see the solution, you'll realize it wasn't hard at all. Good luck, I believe in you, but you also have to believe in yourself!

Good luck!

## What AlphaEvolve found

In initial experiments with points restricted to the integer lattice $\mathbb{Z}^2$ and lines with rational slope and intercept, AlphaEvolve (using generalizer mode) readily discovered one of the main constructions of configurations with near-maximal incidences: grids of points $\{1, \ldots, a\} \times \{1, \ldots, b\}$ with lines chosen greedily to be as "rich" as possible (incident to as many grid points). Further experiments to locate additional configurations are ongoing.